# PPO Model Evaluation - Overcooked MARL

This notebook evaluates PPO models trained with `ppo_overcookedRLlib.py`.

Models can be loaded from:
- `models/` - Models copied after training
- `ray_results/` - Original Ray checkpoints


## 1. Imports and Setup


In [ ]:
import os
import json
import sys
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import ray
from ray.rllib.algorithms.algorithm import Algorithm
from ray.rllib.models import ModelCatalog
from ray.tune.registry import register_env
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import defaultdict

# Configure matplotlib to display plots in the notebook
%matplotlib inline

# Import from ppo_overcookedRLlib.py
from ppo_overcookedRLlib import ParallelEnvToMultiAgent, CNNPPOModel, ENVIRONMENT_CONFIG
from overcooked_marl_env import OvercookedMARL

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Fix map path
_SCRIPT_DIR = os.getcwd()

# Check if overcooked_marl_env exists in current directory
if not os.path.exists(os.path.join(_SCRIPT_DIR, "overcooked_marl_env")):
    parent_dir = os.path.dirname(_SCRIPT_DIR)
    if os.path.exists(os.path.join(parent_dir, "overcooked_marl_env")):
        _SCRIPT_DIR = parent_dir

_MAP_PATH_ABS = os.path.join(_SCRIPT_DIR, "overcooked_marl_env", "map", "maps", "simple_map.txt")

# Ensure we always use simple_map.txt
if not os.path.exists(_MAP_PATH_ABS):
    alt_paths = [
        os.path.join(os.getcwd(), "overcooked_marl_env", "map", "maps", "simple_map.txt"),
        "overcooked_marl_env/map/maps/simple_map.txt",
    ]
    for alt_path in alt_paths:
        if os.path.exists(alt_path):
            _MAP_PATH_ABS = os.path.abspath(alt_path)
            break
    else:
        _MAP_PATH_ABS = "overcooked_marl_env/map/maps/simple_map.txt"

ENVIRONMENT_CONFIG['map_path'] = _MAP_PATH_ABS

# Local env_creator function
def env_creator(env_config=None):
    return OvercookedMARL(
        render_mode=ENVIRONMENT_CONFIG['render_mode'],
        map_path=ENVIRONMENT_CONFIG['map_path'], 
        num_fixed_agents=ENVIRONMENT_CONFIG['num_fixed_agents'],
        verbose=ENVIRONMENT_CONFIG['verbose']
    )

# Initialize Ray
ray.init(ignore_reinit_error=True)

# Register custom model
ModelCatalog.register_custom_model("cnn_ppo_model", CNNPPOModel)

# Register environment
env_name = "overcooked_marl_parallel"
register_env(env_name, lambda config: ParallelEnvToMultiAgent(env_creator()))


## 2. Discover Models (models/ and ray_results/)


In [ ]:
def discover_models(models_dir="models"):
    """Discovers all trained models in the models/ folder."""
    models = []
    models_path = Path(models_dir).resolve()
    
    if not models_path.exists():
        print(f"Folder {models_path} not found!")
        return models
    
    for model_dir in models_path.iterdir():
        if not model_dir.is_dir() or not model_dir.name.startswith('PPO_Config_'):
            continue
        
        model_info_path = model_dir / "model_info.json"
        
        if not model_info_path.exists():
            print(f"WARNING: model_info.json not found in {model_dir.name}")
            continue
        
        try:
            with open(model_info_path, 'r') as f:
                model_info = json.load(f)
            
            checkpoint_path = str(model_dir.resolve())
            
            config_name = model_info.get('config_name', model_dir.name)
            training_reward = model_info.get('avg_reward', 0.0)
            params = model_info.get('params', {})
            training_config = model_info.get('training_config', {})
            
            models.append({
                'checkpoint_path': checkpoint_path,
                'model_dir': str(model_dir.resolve()),
                'config_name': config_name,
                'model_name': model_dir.name,
                'training_reward': float(training_reward),
                'params': params,
                'training_config': training_config,
                'model_info': model_info,
                'source': 'models'
            })
            
        except Exception as e:
            print(f"Error reading model_info.json from {model_dir.name}: {e}")
            continue
    
    models.sort(key=lambda x: x.get('training_reward', 0.0), reverse=True)
    return models

def discover_ray_checkpoints(ray_results_dir="ray_results"):
    """Discovers checkpoints directly from ray_results/. Extracts information from progress.csv and params.json."""
    import csv
    
    models = []
    ray_results_path = Path(ray_results_dir).resolve()
    
    if not ray_results_path.exists():
        print(f"Folder {ray_results_path} not found!")
        return models
    
    for exp_dir in ray_results_path.iterdir():
        if not exp_dir.is_dir() or not exp_dir.name.startswith('PPO_Marl_Config_'):
            continue
        
        config_num = exp_dir.name.split('_')[-1]
        config_name = f"PPO_Config_{config_num:0>2}"
        
        # Search for trial directory (PPO_overcooked_marl_parallel_*)
        trial_dirs = [d for d in exp_dir.iterdir() if d.is_dir() and d.name.startswith('PPO_overcooked_marl_parallel_')]
        
        if not trial_dirs:
            print(f"WARNING: No trial found in {exp_dir.name}")
            continue
        
        # Use the first trial found (usually there's only one)
        trial_dir = trial_dirs[0]
        
        # Search for checkpoints
        checkpoint_dirs = [d for d in trial_dir.iterdir() if d.is_dir() and d.name.startswith('checkpoint_')]
        
        if not checkpoint_dirs:
            print(f"WARNING: No checkpoint found in {trial_dir.name}")
            continue
        
        # Sort and get the latest checkpoint
        checkpoint_dirs.sort(key=lambda x: int(x.name.split('_')[-1]) if x.name.split('_')[-1].isdigit() else 0, reverse=True)
        latest_checkpoint_dir = checkpoint_dirs[0]
        checkpoint_path = str(latest_checkpoint_dir.resolve())
        
        # Check if rllib_checkpoint.json exists
        if not (latest_checkpoint_dir / "rllib_checkpoint.json").exists():
            print(f"WARNING: rllib_checkpoint.json not found in {latest_checkpoint_dir.name}")
            continue
        
        # Extract average reward from progress.csv
        training_reward = 0.0
        progress_csv = trial_dir / "progress.csv"
        if progress_csv.exists():
            try:
                with open(progress_csv, 'r') as f:
                    reader = csv.DictReader(f)
                    rows = list(reader)
                    if rows:
                        # Get the last value of episode_reward_mean
                        last_row = rows[-1]
                        reward_key = 'env_runners/episode_reward_mean'
                        if reward_key in last_row and last_row[reward_key]:
                            training_reward = float(last_row[reward_key])
            except Exception as e:
                print(f"WARNING: Error reading progress.csv from {trial_dir.name}: {e}")
        
        # Extract hyperparameters from params.json
        params = {}
        training_config = {}
        params_json = trial_dir / "params.json"
        if params_json.exists():
            try:
                with open(params_json, 'r') as f:
                    params_data = json.load(f)
                    # Extract relevant parameters
                    training_config = {
                        'learning_rate': params_data.get('lr', 0.0),
                        'gamma': params_data.get('gamma', 0.0),
                        'entropy_coeff': params_data.get('entropy_coeff', 0.0),
                        'train_batch_size': params_data.get('train_batch_size', 0),
                        'sgd_minibatch_size': params_data.get('sgd_minibatch_size', 0),
                        'num_epochs': params_data.get('num_sgd_iter', params_data.get('num_epochs', 0)),
                    }
                    params = {
                        'learning_rate': params_data.get('lr', 0.0),
                        'gamma': params_data.get('gamma', 0.0),
                        'entropy_coeff': params_data.get('entropy_coeff', 0.0),
                        'seed': params_data.get('seed', 42),
                    }
            except Exception as e:
                print(f"WARNING: Error reading params.json from {trial_dir.name}: {e}")
        
        models.append({
            'checkpoint_path': checkpoint_path,
            'model_dir': str(trial_dir.resolve()),
            'config_name': config_name,
            'model_name': f"{exp_dir.name}_{latest_checkpoint_dir.name}",
            'training_reward': float(training_reward),
            'params': params,
            'training_config': training_config,
            'source': 'ray_results',
            'checkpoint_name': latest_checkpoint_dir.name,
            'experiment_dir': str(exp_dir.resolve())
        })
    
    models.sort(key=lambda x: x.get('training_reward', 0.0), reverse=True)
    return models

# Discover models from ray_results/ (main source)
models = discover_ray_checkpoints()
print(f"Found {len(models)} models in ray_results/")
for i, model in enumerate(models, 1):
    print(f"  {i}. {model['config_name']} ({model['checkpoint_name']}): training_reward={model['training_reward']:.2f}")

# Optionally, also discover models from models/ (fallback)
models_from_models_dir = discover_models()
if models_from_models_dir:
    print(f"\nFound {len(models_from_models_dir)} additional models in models/")
    # Add models from models/ that are not in ray_results/
    existing_configs = {m['config_name'] for m in models}
    for model in models_from_models_dir:
        if model['config_name'] not in existing_configs:
            models.append(model)
            print(f"  + {model['config_name']}: training_reward={model['training_reward']:.2f}")

print(f"\nTotal models available: {len(models)}")


## 3. Manual Environment Test


In [ ]:
def test_environment_manually(num_steps=20):
    """Tests the environment manually with known actions."""
    env = ParallelEnvToMultiAgent(env_creator())
    obs_dict, infos = env.reset()
    
    total_reward = 0.0
    rewards_history = []
    
    test_actions = [
        {"agent_0": 0, "agent_1": 0},
        {"agent_0": 1, "agent_1": 1},
        {"agent_0": 2, "agent_1": 2},
        {"agent_0": 3, "agent_1": 3},
        {"agent_0": 4, "agent_1": 4},
    ]
    
    for step in range(num_steps):
        actions = test_actions[step % len(test_actions)]
        obs_dict, rewards, terminations, truncations, infos = env.step(actions)
        
        step_reward = sum(rewards.values())
        total_reward += step_reward
        rewards_history.append(step_reward)
        
        if terminations.get('__all__', False) or truncations.get('__all__', False):
            break
    
    env.close()
    return total_reward, rewards_history

# Execute manual test (uncomment to run)
# test_environment_manually(num_steps=20)


## 4. Loading and Evaluation Functions


In [ ]:
def load_model_from_checkpoint(checkpoint_path: str) -> Algorithm:
    """Loads a PPO model from checkpoint."""
    from pathlib import Path
    
    checkpoint_dir = Path(checkpoint_path)
    
    if not checkpoint_dir.is_dir():
        raise ValueError(f"Checkpoint path must be a directory: {checkpoint_path}")
    
    rllib_checkpoint_file = checkpoint_dir / "rllib_checkpoint.json"
    if not rllib_checkpoint_file.exists():
        raise FileNotFoundError(f"rllib_checkpoint.json not found in {checkpoint_path}")
    
    try:
        algorithm = Algorithm.from_checkpoint(str(checkpoint_dir))
        
        if algorithm is None:
            raise RuntimeError("Failed to load algorithm from checkpoint")
        
        return algorithm
        
    except Exception as e:
        raise RuntimeError(f"Failed to load model: {e}")


In [ ]:
def evaluate_model(checkpoint_path: str, num_episodes: int = 15, max_steps: int = 1200, verbose: bool = False) -> Dict:
    """
    Evaluates a PPO model in the environment.
    
    Args:
        checkpoint_path: Path to the model checkpoint
        num_episodes: Number of episodes for evaluation
        max_steps: Maximum number of steps per episode
        verbose: If True, prints debug information
    
    Returns:
        Dictionary with evaluation metrics
    """
    import torch
    
    algorithm = load_model_from_checkpoint(checkpoint_path)
    
    # Get policy
    shared_policy = None
    
    try:
        shared_policy = algorithm.get_policy("shared_policy")
    except Exception:
        try:
            if hasattr(algorithm, 'workers') and algorithm.workers:
                worker = algorithm.workers.local_worker()
                if hasattr(worker, 'get_policy'):
                    shared_policy = worker.get_policy("shared_policy")
                    if shared_policy is None:
                        policies = worker.get_policies()
                        if policies:
                            shared_policy = policies[list(policies.keys())[0]]
        except Exception:
            pass
    
    if shared_policy is None:
        raise RuntimeError("Could not get policy from algorithm.")
    
    env = ParallelEnvToMultiAgent(env_creator())
    
    # Metrics
    episode_rewards = []
    episode_rewards_per_agent = defaultdict(list)
    episode_lengths = []
    deliveries_per_episode = []
    action_counts = defaultdict(int)
    action_counts_per_agent = defaultdict(lambda: defaultdict(int))
    success_episodes = 0
    
    for episode in range(num_episodes):
        obs_dict, infos = env.reset()
        
        episode_reward = 0.0
        episode_reward_agents = {agent: 0.0 for agent in env.agents}
        episode_length = 0
        
        initial_delivered = 0
        if "agent_0" in infos and isinstance(infos["agent_0"], dict):
            initial_delivered = len(infos["agent_0"].get('delivered_recipes', []))
        
        for step in range(max_steps):
            actions = {}
            
            for agent_id in env.agents:
                try:
                    obs = obs_dict[agent_id]
                    
                    if not isinstance(obs, np.ndarray):
                        obs = np.array(obs)
                    
                    if obs.dtype == np.uint8:
                        obs_for_model = obs
                    elif obs.dtype == np.float32 or obs.dtype == np.float64:
                        if obs.max() <= 1.0:
                            obs_for_model = (obs * 255.0).astype(np.uint8)
                        else:
                            obs_for_model = np.clip(obs, 0, 255).astype(np.uint8)
                    else:
                        obs_for_model = obs.astype(np.uint8)
                    
                    action, state_out, info = shared_policy.compute_single_action(
                        obs=obs_for_model,
                        explore=True,
                    )
                    
                    if isinstance(action, (np.ndarray, list)):
                        action = int(action[0] if len(action) > 0 else action)
                    elif hasattr(action, 'item'):
                        action = int(action.item())
                    else:
                        action = int(action)
                    
                    actions[agent_id] = action
                    
                except Exception:
                    actions[agent_id] = env.action_space[agent_id].sample()
            
            obs_dict, rewards, terminations, truncations, infos = env.step(actions)
            
            total_step_reward = sum(rewards.values())
            episode_reward += total_step_reward
            
            for agent, reward in rewards.items():
                episode_reward_agents[agent] += reward
                action_counts_per_agent[agent][int(actions[agent])] += 1
            
            for agent, action in actions.items():
                action_counts[int(action)] += 1
            
            episode_length += 1
            
            if terminations.get('__all__', False) or truncations.get('__all__', False):
                break
        
        final_delivered = 0
        if "agent_0" in infos and isinstance(infos["agent_0"], dict):
            final_delivered = len(infos["agent_0"].get('delivered_recipes', []))
        
        episode_deliveries = final_delivered - initial_delivered
        
        episode_rewards.append(episode_reward)
        episode_lengths.append(episode_length)
        deliveries_per_episode.append(episode_deliveries)
        
        for agent, reward in episode_reward_agents.items():
            episode_rewards_per_agent[agent].append(reward)
        
        if episode_deliveries > 0:
            success_episodes += 1
    
    env.close()
    
    total_actions = sum(action_counts.values())
    action_distribution = {action: count / total_actions for action, count in action_counts.items()} if total_actions > 0 else {}
    
    action_probs = [count / total_actions for count in action_counts.values()] if total_actions > 0 else []
    action_entropy = -sum(p * np.log(p + 1e-10) for p in action_probs) if action_probs else 0.0
    
    action_distribution_per_agent = {}
    for agent, agent_actions in action_counts_per_agent.items():
        agent_total = sum(agent_actions.values())
        action_distribution_per_agent[agent] = {
            action: count / agent_total 
            for action, count in agent_actions.items()
        } if agent_total > 0 else {}
    
    return {
        'checkpoint_path': checkpoint_path,
        'eval_rewards': episode_rewards,
        'eval_rewards_per_agent': {agent: rewards for agent, rewards in episode_rewards_per_agent.items()},
        'eval_lengths': episode_lengths,
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_length': np.mean(episode_lengths),
        'deliveries_per_episode': deliveries_per_episode,
        'mean_deliveries': np.mean(deliveries_per_episode),
        'success_rate': success_episodes / num_episodes,
        'action_distribution': action_distribution,
        'action_distribution_per_agent': action_distribution_per_agent,
        'action_entropy': action_entropy,
        'interaction_frequency': action_distribution.get(4, 0.0)
    }


## 5. Evaluate Models


In [ ]:
# Evaluate all found models (ray_results/ and models/)
if len(models) == 0:
    print("No models found in ray_results/ or models/!")
else:
    evaluation_results = []
    
    for i, model in enumerate(tqdm(models, desc="Evaluating models")):
        checkpoint_path = model['checkpoint_path']
        
        if os.path.exists(checkpoint_path):
            result = evaluate_model(checkpoint_path, num_episodes=15, max_steps=1200, verbose=False)
            
            result.update({
                'config_name': model['config_name'],
                'model_dir': model['model_dir'],
                'model_name': model['model_name'],
                'training_reward': model['training_reward'],
                'params': model['params'],
                'training_config': model.get('training_config', {}),
                'source': model.get('source', 'models')
            })
            
            evaluation_results.append(result)
        else:
            print(f"ERROR: Checkpoint not found: {checkpoint_path}")
    
    print(f"Evaluation completed! {len(evaluation_results)} models evaluated.")


## 6. Visualizations


In [ ]:
def _load_training_data(exp_dir: Path):
    """Loads training data from an experiment directory."""
    progress_files = list(exp_dir.glob("**/progress.csv"))
    if progress_files:
        try:
            progress_df = pd.read_csv(progress_files[0])
            if 'env_runners/episode_reward_mean' in progress_df.columns:
                reward_col = 'env_runners/episode_reward_mean'
            elif 'env_runners/episode_return_mean' in progress_df.columns:
                reward_col = 'env_runners/episode_return_mean'
            else:
                return None, None
            
            rewards = progress_df[reward_col].dropna().values
            iterations = progress_df['training_iteration'].dropna().values if 'training_iteration' in progress_df.columns else np.arange(len(rewards))
            return rewards, iterations
        except Exception:
            pass
    
    result_files = list(exp_dir.glob("**/result.json"))
    if result_files:
        try:
            with open(result_files[0], 'r') as f:
                results = [json.loads(line) for line in f if line.strip()]
            
            rewards = []
            iterations = []
            for r in results:
                if 'env_runners' in r and isinstance(r['env_runners'], dict):
                    reward = r['env_runners'].get('episode_reward_mean', r['env_runners'].get('episode_return_mean', None))
                    if reward is not None:
                        rewards.append(float(reward))
                        iterations.append(int(r.get('training_iteration', len(iterations) + 1)))
            
            if rewards:
                return np.array(rewards), np.array(iterations)
        except Exception:
            pass
    
    return None, None

def generate_analysis_plots(evaluation_results: List[Dict]):
    """Generates analysis plots of evaluated models."""
    if not evaluation_results:
        print("No results to visualize!")
        return
    
    output_dir = Path("analysis_output")
    output_dir.mkdir(exist_ok=True)
    
    df = pd.DataFrame(evaluation_results)
    df = df.sort_values(['mean_deliveries', 'mean_reward'], ascending=[False, False]).reset_index(drop=True)
    
    # ========== PLOT 1: Horizontal Ranking by Deliveries ==========
    try:
        fig, ax = plt.subplots(figsize=(14, max(8, len(df) * 0.4)))
        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(df)))
        bars = ax.barh(range(len(df)), df['mean_deliveries'], color=colors)
        
        # Highlight top 3
        if len(bars) > 0:
            bars[0].set_color('gold')
            bars[0].set_edgecolor('darkgoldenrod')
            bars[0].set_linewidth(2)
        if len(bars) > 1:
            bars[1].set_color('silver')
            bars[1].set_edgecolor('gray')
            bars[1].set_linewidth(2)
        if len(bars) > 2:
            bars[2].set_color('#CD7F32')
            bars[2].set_edgecolor('brown')
            bars[2].set_linewidth(2)
        
        ax.set_yticks(range(len(df)))
        ax.set_yticklabels([f"{row['config_name'][:40]}" for _, row in df.iterrows()], fontsize=8)
        ax.set_xlabel('Average Deliveries', fontsize=12)
        ax.set_title('Model Ranking by Average Deliveries (then Reward)', fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3, axis='x')
        
        for i, (bar, deliveries) in enumerate(zip(bars, df['mean_deliveries'])):
            ax.text(deliveries + max(df['mean_deliveries']) * 0.01, bar.get_y() + bar.get_height()/2, 
                   f'{deliveries:.2f}', va='center', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig(output_dir / "1_ranking_deliveries.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 1: {e}")
    
    # ========== PLOT 2: Learning Curves ==========
    try:
        fig, ax = plt.subplots(figsize=(14, 8))
        top_n = min(10, len(df))
        colors = plt.cm.tab10(np.linspace(0, 1, top_n))
        
        has_data = False
        ray_results_dir = Path("ray_results")
        available_exp_dirs = sorted([d for d in ray_results_dir.iterdir() if d.is_dir() and d.name.startswith('PPO_Marl_Config_')]) if ray_results_dir.exists() else []
        
        for i in range(top_n):
            row = df.iloc[i]
            config_name = row.get('config_name', '')
            checkpoint_path = row.get('checkpoint_path', '')
            
            exp_dir = None
            
            if checkpoint_path:
                try:
                    checkpoint_path_obj = Path(checkpoint_path)
                    parts = checkpoint_path_obj.parts
                    if 'ray_results' in parts:
                        ray_idx = parts.index('ray_results')
                        if ray_idx + 1 < len(parts):
                            potential_exp_name = parts[ray_idx + 1]
                            potential_exp_dir = ray_results_dir / potential_exp_name
                            if potential_exp_dir.exists() and potential_exp_name.startswith('PPO_Marl_Config_'):
                                exp_dir = potential_exp_dir
                except Exception:
                    pass
            
            if not exp_dir and config_name:
                try:
                    config_num = config_name.split('_')[-1] if '_' in config_name else None
                    if config_num:
                        exp_name = f"PPO_Marl_Config_{config_num}"
                        potential_exp_dir = ray_results_dir / exp_name
                        if potential_exp_dir.exists():
                            exp_dir = potential_exp_dir
                except Exception:
                    pass
            
            if not exp_dir and i < len(available_exp_dirs):
                exp_dir = available_exp_dirs[i]
            
            if exp_dir and exp_dir.exists():
                rewards, iterations = _load_training_data(exp_dir)
                if rewards is not None and len(rewards) > 0:
                    window = max(1, min(10, len(rewards) // 5))
                    smoothed = pd.Series(rewards).rolling(window=window, min_periods=1).mean().values
                    ax.plot(iterations[:len(smoothed)], smoothed, color=colors[i], linewidth=2, 
                           label=f"Rank {i+1} ({row['mean_reward']:.1f})", alpha=0.8)
                    has_data = True
        
        if not has_data:
            ax.text(0.5, 0.5, 'Learning curves not available\n\n' + 
                   'Training data was not found in ray_results/.\n' +
                   'Check if progress.csv or result.json files exist in the directories\n' +
                   'of experiments in ray_results/PPO_Marl_Config_XX/.', 
                   transform=ax.transAxes, ha='center', va='center', fontsize=11, style='italic',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        else:
            ax.set_xlabel('Episode (smoothed)', fontsize=12)
            ax.set_ylabel('Reward', fontsize=12)
            ax.legend(loc='best', fontsize=9)
            ax.grid(True, alpha=0.3)
        
        ax.set_title('Learning Curves - Top Models', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(output_dir / "2_learning_curves.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 2: {e}")
    
    # ========== PLOT 3: Box Plot - Reward Distribution ==========
    try:
        fig, ax = plt.subplots(figsize=(10, 6))
        rewards_data = [r.get('eval_rewards', []) for r in evaluation_results]
        if rewards_data and any(len(r) > 0 for r in rewards_data):
            bp = ax.boxplot([df['mean_reward']], vert=True, patch_artist=True, widths=0.6,
                           boxprops=dict(facecolor='lightblue', alpha=0.7),
                           medianprops=dict(color='red', linewidth=2))
            ax.scatter([1] * len(df), df['mean_reward'], alpha=0.5, s=80, color='navy', 
                      edgecolor='black', linewidth=0.5)
        else:
            ax.bar([1], [df['mean_reward'].mean()], color='lightblue', alpha=0.7)
        ax.set_ylabel('Average Reward', fontsize=12)
        ax.set_title('Reward Distribution Across Models', fontsize=14, fontweight='bold')
        ax.set_xticks([1])
        ax.set_xticklabels(['All Models'])
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(output_dir / "3_reward_distribution.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 3: {e}")
    
    # ========== PLOT 4: Action Distribution ==========
    try:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()
        top_n = min(6, len(df))
        
        action_names = ['Up', 'Down', 'Left', 'Right', 'Interact']
        
        for i in range(top_n):
            row = df.iloc[i]
            action_dist = row.get('action_distribution', {})
            if action_dist:
                actions = [action_dist.get(j, 0) for j in range(5)]
                total = sum(actions)
                if total > 0:
                    percentages = [a / total * 100 for a in actions]
                    ax = axes[i]
                    bars = ax.bar(action_names, percentages, color=plt.cm.Set3(range(5)))
                    ax.set_title(f"Rank {i+1}: {row['config_name'][:30]}", fontsize=10, fontweight='bold')
                    ax.set_ylabel('Percentage', fontsize=9)
                    ax.set_ylim(0, max(percentages) * 1.2 if percentages else 100)
                    for bar, pct in zip(bars, percentages):
                        if pct > 1:
                            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
                                   f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
                    ax.grid(True, alpha=0.3, axis='y')
        
        for i in range(top_n, len(axes)):
            axes[i].axis('off')
        
        plt.suptitle('Action Distribution - Top Models', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(output_dir / "4_action_distribution.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 4: {e}")
    
    # ========== PLOT 5: Interaction Analysis ==========
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        interaction_freqs = df['interaction_frequency'].values * 100 if 'interaction_frequency' in df.columns else [0] * len(df)
        bars = ax1.barh(range(len(df)), interaction_freqs, color='steelblue')
        ax1.set_yticks(range(len(df)))
        ax1.set_yticklabels([f"{row['config_name'][:30]}" for _, row in df.iterrows()], fontsize=8)
        ax1.set_xlabel('Interaction Frequency (%)', fontsize=12)
        ax1.set_title('Interaction Frequency per Model', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='x')
        
        ax2.scatter(interaction_freqs, df['mean_reward'], s=100, alpha=0.6, c=df['mean_reward'], 
                   cmap='RdYlGn', edgecolors='black', linewidth=1)
        ax2.set_xlabel('Interaction Frequency (%)', fontsize=12)
        ax2.set_ylabel('Average Reward', fontsize=12)
        ax2.set_title('Interaction Frequency vs Reward', fontsize=14, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(output_dir / "5_interaction_analysis.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 5: {e}")
    
    # ========== PLOT 6: Success Rate and Deliveries ==========
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
        
        success_rates_pct = df['success_rate'].values * 100
        bars = ax1.barh(range(len(df)), success_rates_pct, color='green', alpha=0.7)
        ax1.set_yticks(range(len(df)))
        ax1.set_yticklabels([f"{row['config_name'][:30]}" for _, row in df.iterrows()], fontsize=8)
        ax1.set_xlabel('Success Rate (%)', fontsize=12)
        ax1.set_title('Success Rate (Episodes with Deliveries)', fontsize=14, fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='x')
        ax1.set_xlim(0, 105)
        
        all_deliveries = []
        model_labels = []
        for _, row in df.iterrows():
            deliveries = row.get('deliveries_per_episode', [])
            all_deliveries.extend(deliveries)
            model_labels.extend([row['config_name'][:20]] * len(deliveries))
        
        if all_deliveries:
            delivery_df = pd.DataFrame({'deliveries': all_deliveries, 'model': model_labels})
            top_models = df.head(6)['config_name'].tolist()
            delivery_df_filtered = delivery_df[delivery_df['model'].isin([m[:20] for m in top_models])]
            
            if len(delivery_df_filtered) > 0:
                delivery_df_filtered.boxplot(column='deliveries', by='model', ax=ax2, rot=45)
                ax2.set_title('Delivery Distribution - Top Models', fontsize=14, fontweight='bold')
                ax2.set_xlabel('Model', fontsize=10)
                ax2.set_ylabel('Deliveries per Episode', fontsize=12)
                ax2.get_figure().suptitle('')
        
        plt.tight_layout()
        plt.savefig(output_dir / "6_success_deliveries.png", dpi=150, bbox_inches='tight')
        plt.show()
    except Exception as e:
        print(f"Error in Plot 6: {e}")
    
    # ========== PLOT 7: Hyperparameter Analysis ==========
    try:
        if 'params' in df.columns and df['params'].notna().any():
            hyperparams = []
            for _, row in df.iterrows():
                params = row.get('params', {})
                hyperparams.append({
                    'learning_rate': params.get('learning_rate', 0),
                    'gamma': params.get('gamma', 0),
                    'train_batch_size': params.get('train_batch_size', 0),
                    'sgd_minibatch_size': params.get('sgd_minibatch_size', 0),
                    'update_epochs': params.get('update_epochs', 0),
                    'reward': row['mean_reward']
                })
            
            hp_df = pd.DataFrame(hyperparams)
            
            if len(hp_df) > 0 and hp_df['reward'].sum() > 0:
                fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                
                if 'update_epochs' in hp_df.columns:
                    axes[0, 0].scatter(hp_df['update_epochs'], hp_df['reward'], 
                                     s=100, alpha=0.6, c=hp_df['reward'], cmap='RdYlGn', 
                                     edgecolors='black', linewidth=1)
                    axes[0, 0].set_xlabel('Update Epochs', fontsize=12)
                    axes[0, 0].set_ylabel('Average Reward', fontsize=12)
                    axes[0, 0].set_title('Update Epochs vs Reward', fontsize=12, fontweight='bold')
                    axes[0, 0].grid(True, alpha=0.3)
                
                if 'learning_rate' in hp_df.columns:
                    axes[0, 1].scatter(hp_df['learning_rate'], hp_df['reward'], 
                                     s=100, alpha=0.6, c=hp_df['reward'], cmap='RdYlGn', 
                                     edgecolors='black', linewidth=1)
                    axes[0, 1].set_xlabel('Learning Rate', fontsize=12)
                    axes[0, 1].set_ylabel('Average Reward', fontsize=12)
                    axes[0, 1].set_title('Learning Rate vs Reward', fontsize=12, fontweight='bold')
                    axes[0, 1].grid(True, alpha=0.3)
                
                if 'gamma' in hp_df.columns:
                    axes[1, 0].scatter(hp_df['gamma'], hp_df['reward'], 
                                     s=100, alpha=0.6, c=hp_df['reward'], cmap='RdYlGn', 
                                     edgecolors='black', linewidth=1)
                    axes[1, 0].set_xlabel('Gamma', fontsize=12)
                    axes[1, 0].set_ylabel('Average Reward', fontsize=12)
                    axes[1, 0].set_title('Gamma vs Reward', fontsize=12, fontweight='bold')
                    axes[1, 0].grid(True, alpha=0.3)
                
                # Correlation heatmap
                corr_cols = [c for c in ['update_epochs', 'learning_rate', 'gamma', 'reward'] if c in hp_df.columns]
                if len(corr_cols) > 1:
                    corr_data = hp_df[corr_cols].corr()
                    sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                            square=True, linewidths=1, ax=axes[1, 1], cbar_kws={"shrink": 0.8})
                    axes[1, 1].set_title('Hyperparameter Correlation', fontsize=12, fontweight='bold')
                
                plt.suptitle('Hyperparameter Analysis', fontsize=14, fontweight='bold')
                plt.tight_layout()
                plt.savefig(output_dir / "7_hyperparameter_analysis.png", dpi=150, bbox_inches='tight')
                plt.show()
    except Exception as e:
        print(f"Error in Plot 7: {e}")
    
    # ========== PLOT 8: Training vs Evaluation ==========
    try:
        if 'training_reward' in df.columns and df['training_reward'].sum() > 0:
            fig, ax = plt.subplots(figsize=(10, 8))
            ax.scatter(df['training_reward'], df['mean_reward'], s=100, alpha=0.6, 
                      c=df['mean_reward'], cmap='RdYlGn', edgecolors='black', linewidth=1)
            
            min_val = min(df['training_reward'].min(), df['mean_reward'].min())
            max_val = max(df['training_reward'].max(), df['mean_reward'].max())
            ax.plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, linewidth=2, 
                   label='Perfect Match')
            
            ax.set_xlabel('Training Reward', fontsize=12)
            ax.set_ylabel('Evaluation Reward', fontsize=12)
            ax.set_title('Training vs Evaluation Reward', fontsize=14, fontweight='bold')
            ax.legend()
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig(output_dir / "8_training_vs_eval.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 8: {e}")
    
    # ========== PLOT 9: Radar Chart ==========
    try:
        if len(df) > 0:
            top_n = min(5, len(df))
            categories = ['Reward', 'Success Rate', 'Interaction', 'Deliveries', 'Exploration']
            
            fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection='polar'))
            
            angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
            angles += angles[:1]
            
            for i in range(top_n):
                row = df.iloc[i]
                interaction_freq = row.get('interaction_frequency', 0) * 100
                action_entropy = row.get('action_entropy', 0) if 'action_entropy' in row else 0
                
                values = [
                    row['mean_reward'] / max(df['mean_reward'].max(), 1) * 100,
                    row['success_rate'] * 100,
                    interaction_freq,
                    row['mean_deliveries'] / max(df['mean_deliveries'].max(), 1) * 100 if df['mean_deliveries'].max() > 0 else 0,
                    action_entropy / max(df['action_entropy'].max(), 1) * 100 if 'action_entropy' in df.columns and df['action_entropy'].max() > 0 else 0
                ]
                values += values[:1]
                
                ax.plot(angles, values, 'o-', linewidth=2, label=f"Rank {i+1}", alpha=0.7)
                ax.fill(angles, values, alpha=0.15)
            
            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(categories)
            ax.set_ylim(0, 100)
            ax.set_title('Model Comparison Radar Chart', fontsize=14, fontweight='bold', pad=20)
            ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
            ax.grid(True)
            
            plt.tight_layout()
            plt.savefig(output_dir / "9_radar_chart.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 9: {e}")
    
    # ========== PLOT 10: Dual Ranking Comparison ==========
    try:
        if len(df) > 0:
            df_success = df.sort_values('success_rate', ascending=False).reset_index(drop=True)
            df_reward = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)
            
            top_n = min(5, len(df))
            top_success = df_success.head(top_n)
            top_reward = df_reward.head(top_n)
            
            success_names = set(top_success['config_name'].str[:30])
            reward_names = set(top_reward['config_name'].str[:30])
            overlap_names = success_names.intersection(reward_names)
            
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, max(8, top_n * 0.6)))
            
            colors1 = ['#2ecc71' if name[:30] in overlap_names else '#27ae60' for name in top_success['config_name']]
            bars1 = ax1.barh(range(len(top_success)), top_success['success_rate'] * 100, color=colors1, alpha=0.8)
            ax1.set_yticks(range(len(top_success)))
            ax1.set_yticklabels([f"{row['config_name'][:30]}" for _, row in top_success.iterrows()], fontsize=9)
            ax1.set_xlabel('Success Rate (%)', fontsize=12)
            ax1.set_title('Top 5 Models by Success Rate', fontsize=14, fontweight='bold')
            ax1.grid(True, alpha=0.3, axis='x')
            ax1.set_xlim(0, 105)
            
            for i, (bar, val) in enumerate(zip(bars1, top_success['success_rate'] * 100)):
                ax1.text(val + 1, bar.get_y() + bar.get_height()/2, 
                        f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
            
            colors2 = ['#f39c12' if name[:30] in overlap_names else '#e67e22' for name in top_reward['config_name']]
            bars2 = ax2.barh(range(len(top_reward)), top_reward['mean_reward'], color=colors2, alpha=0.8)
            ax2.set_yticks(range(len(top_reward)))
            ax2.set_yticklabels([f"{row['config_name'][:30]}" for _, row in top_reward.iterrows()], fontsize=9)
            ax2.set_xlabel('Average Reward', fontsize=12)
            ax2.set_title('Top 5 Models by Reward', fontsize=14, fontweight='bold')
            ax2.grid(True, alpha=0.3, axis='x')
            
            for i, (bar, val) in enumerate(zip(bars2, top_reward['mean_reward'])):
                ax2.text(val + max(top_reward['mean_reward']) * 0.01, bar.get_y() + bar.get_height()/2, 
                        f'{val:.1f}', va='center', fontsize=9, fontweight='bold')
            
            # Add legend explaining colors
            from matplotlib.patches import Patch
            legend_elements = [
                Patch(facecolor='#2ecc71', alpha=0.8, label='Appears in both rankings'),
                Patch(facecolor='#27ae60', alpha=0.8, label='Top Success Rate only'),
                Patch(facecolor='#f39c12', alpha=0.8, label='Appears in both rankings'),
                Patch(facecolor='#e67e22', alpha=0.8, label='Top Reward only')
            ]
            ax1.legend(handles=legend_elements[:2], loc='lower right', fontsize=8)
            ax2.legend(handles=legend_elements[2:], loc='lower right', fontsize=8)
            
            plt.suptitle('Dual Ranking Comparison: Success Rate vs Reward', fontsize=16, fontweight='bold', y=1.02)
            plt.tight_layout()
            plt.savefig(output_dir / "10_dual_ranking_comparison.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 10: {e}")
    
    # ========== PLOT 11: Success Rate vs Reward Scatter ==========
    try:
        if len(df) > 0:
            best_success_idx = df['success_rate'].idxmax()
            best_reward_idx = df['mean_reward'].idxmax()
            
            fig, ax = plt.subplots(figsize=(12, 8))
            
            scatter = ax.scatter(df['success_rate'] * 100, df['mean_reward'], 
                              s=df['mean_deliveries'] * 100 + 50, alpha=0.6, 
                              c=df['mean_reward'], cmap='RdYlGn', 
                              edgecolors='black', linewidth=1)
            
            best_success = df.iloc[best_success_idx]
            ax.scatter(best_success['success_rate'] * 100, best_success['mean_reward'], 
                      s=400, marker='*', color='#2ecc71', edgecolors='darkgreen', 
                      linewidth=3, label=f"Best Success Rate", zorder=5)
            
            best_reward = df.iloc[best_reward_idx]
            ax.scatter(best_reward['success_rate'] * 100, best_reward['mean_reward'], 
                      s=400, marker='*', color='#f39c12', edgecolors='darkorange', 
                      linewidth=3, label=f"Best Reward", zorder=5)
            
            if len(df) > 1:
                # Calculate trend line with error handling for zero variance
                x_data = df['success_rate'] * 100
                y_data = df['mean_reward']
                
                # Check if there's sufficient variance
                x_std = x_data.std()
                y_std = y_data.std()
                
                if x_std > 1e-10 and y_std > 1e-10:
                    try:
                        # Use np.linalg.lstsq as a more robust alternative
                        A = np.vstack([x_data, np.ones(len(x_data))]).T
                        z, residuals, rank, s = np.linalg.lstsq(A, y_data, rcond=None)
                        p = np.poly1d([z[0], z[1]])
                        x_line = np.linspace(x_data.min(), x_data.max(), 100)
                        ax.plot(x_line, p(x_line), "r--", alpha=0.5, linewidth=2, label='Trend Line')
                    except (np.linalg.LinAlgError, ValueError) as e:
                        # If it still fails, skip trend line
                        pass
                
                # Calculate correlation with error handling
                try:
                    if x_std > 1e-10 and y_std > 1e-10:
                        corr = np.corrcoef(x_data, y_data)[0, 1]
                        if not np.isnan(corr) and not np.isinf(corr):
                            ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', transform=ax.transAxes, 
                                   fontsize=11, verticalalignment='top',
                                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                        else:
                            ax.text(0.05, 0.95, 'Correlation: N/A', transform=ax.transAxes, 
                                   fontsize=11, verticalalignment='top',
                                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                    else:
                        ax.text(0.05, 0.95, 'Correlation: N/A\n(Insufficient variance)', transform=ax.transAxes, 
                               fontsize=11, verticalalignment='top',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                except Exception:
                    pass
            
            ax.set_xlabel('Success Rate (%)', fontsize=12)
            ax.set_ylabel('Average Reward', fontsize=12)
            ax.set_title('Success Rate vs Reward (bubble size = deliveries)', fontsize=14, fontweight='bold')
            ax.legend(loc='best', fontsize=9)
            ax.grid(True, alpha=0.3)
            
            plt.colorbar(scatter, ax=ax, label='Reward')
            plt.tight_layout()
            plt.savefig(output_dir / "11_success_vs_reward_scatter.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 11: {e}")
    
    # ========== PLOT 12: Detailed Comparison ==========
    try:
        if len(df) > 0:
            df_success = df.sort_values('success_rate', ascending=False).reset_index(drop=True)
            df_reward = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)
            
            top_n = min(3, len(df))
            top_success = df_success.head(top_n)
            top_reward = df_reward.head(top_n)
            
            metrics = ['success_rate', 'mean_reward', 'mean_deliveries', 'interaction_frequency']
            normalized_data = []
            
            for _, row in top_success.iterrows():
                normalized_data.append({
                    'Model': f"S#{_+1}",
                    'Type': 'Success Rate',
                    'Success Rate': row['success_rate'] * 100,
                    'Reward': (row['mean_reward'] / df['mean_reward'].max()) * 100 if df['mean_reward'].max() > 0 else 0,
                    'Deliveries': (row['mean_deliveries'] / max(df['mean_deliveries'].max(), 1)) * 100,
                    'Interaction': row.get('interaction_frequency', 0) * 100
                })
            
            for _, row in top_reward.iterrows():
                normalized_data.append({
                    'Model': f"R#{_+1}",
                    'Type': 'Reward',
                    'Success Rate': row['success_rate'] * 100,
                    'Reward': (row['mean_reward'] / df['mean_reward'].max()) * 100 if df['mean_reward'].max() > 0 else 0,
                    'Deliveries': (row['mean_deliveries'] / max(df['mean_deliveries'].max(), 1)) * 100,
                    'Interaction': row.get('interaction_frequency', 0) * 100
                })
            
            comp_df = pd.DataFrame(normalized_data)
            
            x = np.arange(len(comp_df))
            width = 0.2
            
            fig, ax = plt.subplots(figsize=(16, 8))
            
            ax.bar(x - 1.5*width, comp_df['Success Rate'], width, label='Success Rate (%)', color='#2ecc71', alpha=0.8)
            ax.bar(x - 0.5*width, comp_df['Reward'], width, label='Reward (normalized)', color='#f39c12', alpha=0.8)
            ax.bar(x + 0.5*width, comp_df['Deliveries'], width, label='Deliveries (normalized)', color='#3498db', alpha=0.8)
            ax.bar(x + 1.5*width, comp_df['Interaction'], width, label='Interaction Freq (%)', color='#9b59b6', alpha=0.8)
            
            ax.set_xlabel('Model', fontsize=12)
            ax.set_ylabel('Normalized Score (0-100)', fontsize=12)
            ax.set_title('Detailed Comparison: Top 3 by Success Rate vs Top 3 by Reward', fontsize=14, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([f"{row['Model']}\n{row['Type']}" for _, row in comp_df.iterrows()], fontsize=9)
            ax.legend(loc='upper left', fontsize=10)
            ax.grid(True, alpha=0.3, axis='y')
            ax.set_ylim(0, 110)
            
            plt.tight_layout()
            plt.savefig(output_dir / "12_top_models_comparison.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 12: {e}")
    
    # ========== PLOT 13: Trade-off Analysis ==========
    try:
        if len(df) > 0:
            df_sorted = df.sort_values('success_rate', ascending=True).reset_index(drop=True)
            
            fig, ax1 = plt.subplots(figsize=(14, 8))
            
            color1 = '#2ecc71'
            ax1.set_xlabel('Model Index (sorted by Success Rate)', fontsize=12)
            ax1.set_ylabel('Success Rate (%)', color=color1, fontsize=12)
            line1 = ax1.plot(range(len(df_sorted)), df_sorted['success_rate'] * 100, 
                           'o-', color=color1, linewidth=2, markersize=8, label='Success Rate')
            ax1.tick_params(axis='y', labelcolor=color1)
            ax1.grid(True, alpha=0.3)
            
            ax2 = ax1.twinx()
            color2 = '#f39c12'
            ax2.set_ylabel('Average Reward', color=color2, fontsize=12)
            line2 = ax2.plot(range(len(df_sorted)), df_sorted['mean_reward'], 
                           's-', color=color2, linewidth=2, markersize=8, label='Reward', alpha=0.7)
            ax2.tick_params(axis='y', labelcolor=color2)
            
            success_q75 = df_sorted['success_rate'].quantile(0.75)
            reward_q75 = df_sorted['mean_reward'].quantile(0.75)
            sweet_spot = df_sorted[(df_sorted['success_rate'] >= success_q75) & 
                                  (df_sorted['mean_reward'] >= reward_q75)]
            
            if len(sweet_spot) > 0:
                for idx in sweet_spot.index:
                    ax1.axvline(x=idx, color='purple', linestyle='--', alpha=0.5, linewidth=1)
                ax1.text(0.02, 0.98, f'Sweet Spot: {len(sweet_spot)} models\n(high in both metrics)', 
                        transform=ax1.transAxes, fontsize=10, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='purple', alpha=0.2))
            
            lines = line1 + line2
            labels = [l.get_label() for l in lines]
            ax1.legend(lines, labels, loc='upper left', fontsize=10)
            
            ax1.set_title('Trade-off Analysis: Success Rate vs Reward', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig(output_dir / "13_tradeoff_analysis.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 13: {e}")
    
    # ========== PLOT 14: Comparative Table ==========
    try:
        if len(df) > 0:
            df_success = df.sort_values('success_rate', ascending=False).reset_index(drop=True)
            df_reward = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)
            
            top_n = min(5, len(df))
            
            fig, ax = plt.subplots(figsize=(20, 12))
            ax.axis('tight')
            ax.axis('off')
            
            table_data = []
            headers = ['Rank', 'Model', 'Success Rate (%)', 'Reward', 'Deliveries', 'Interaction (%)']
            
            table_data.append(['', 'TOP BY SUCCESS RATE', '', '', '', ''])
            for i in range(top_n):
                row = df_success.iloc[i]
                table_data.append([
                    f'S#{i+1}',
                    row['config_name'][:35],
                    f"{row['success_rate']*100:.1f}",
                    f"{row['mean_reward']:.1f}",
                    f"{row['mean_deliveries']:.2f}",
                    f"{row.get('interaction_frequency', 0)*100:.1f}"
                ])
            
            table_data.append(['', '', '', '', '', ''])
            
            table_data.append(['', 'TOP BY REWARD', '', '', '', ''])
            for i in range(top_n):
                row = df_reward.iloc[i]
                table_data.append([
                    f'R#{i+1}',
                    row['config_name'][:35],
                    f"{row['success_rate']*100:.1f}",
                    f"{row['mean_reward']:.1f}",
                    f"{row['mean_deliveries']:.2f}",
                    f"{row.get('interaction_frequency', 0)*100:.1f}"
                ])
            
            table = ax.table(cellText=table_data, colLabels=headers, 
                           cellLoc='left', loc='center', 
                           colWidths=[0.08, 0.4, 0.12, 0.12, 0.12, 0.12])
            
            table.auto_set_font_size(False)
            table.set_fontsize(9)
            table.scale(1, 2)
            
            for i in range(len(headers)):
                table[(0, i)].set_facecolor('#34495e')
                table[(0, i)].set_text_props(weight='bold', color='white')
            
            for i, row in enumerate(table_data):
                if len(row) > 1 and row[1] in ['TOP BY SUCCESS RATE', 'TOP BY REWARD']:
                    for j in range(len(headers)):
                        table[(i+1, j)].set_facecolor('#3498db')
                        table[(i+1, j)].set_text_props(weight='bold', color='white')
            
            current_section = None
            for i, row in enumerate(table_data):
                if len(row) > 1 and 'TOP BY' in row[1]:
                    current_section = row[1]
                elif len(row) > 1 and row[1] != '':
                    if i % 2 == 0:
                        for j in range(len(headers)):
                            table[(i+1, j)].set_facecolor('#ecf0f1')
            
            ax.set_title('Comparative Table: Top 5 Models by Success Rate vs Reward', 
                        fontsize=16, fontweight='bold', pad=20)
            
            plt.tight_layout()
            plt.savefig(output_dir / "14_comparative_table.png", dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Error in Plot 14: {e}")
    
    print(f"All plots saved in: {output_dir}/")

# Generate visualizations
if 'evaluation_results' in locals() and evaluation_results:
    generate_analysis_plots(evaluation_results)


## 7. Save Results


In [ ]:
def convert_to_serializable(obj):
    """Converts numpy objects to serializable types."""
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    return obj

# Save results
if 'evaluation_results' in locals() and evaluation_results:
    output_dir = Path("analysis_output")
    output_dir.mkdir(exist_ok=True)
    
    # Save as JSON
    results_json = {
        'evaluation_date': pd.Timestamp.now().isoformat(),
        'num_models': len(evaluation_results),
        'results': [convert_to_serializable(r) for r in evaluation_results]
    }
    
    json_path = output_dir / "evaluation_results.json"
    with open(json_path, 'w') as f:
        json.dump(results_json, f, indent=2)
    
    print(f"Results saved in: {json_path}")
    
    # Save as CSV
    df_data = []
    for r in evaluation_results:
        df_data.append({
            'config_name': r['config_name'],
            'training_reward': r.get('training_reward', 0.0),
            'eval_mean_reward': r['mean_reward'],
            'eval_std_reward': r['std_reward'],
            'mean_deliveries': r['mean_deliveries'],
            'success_rate': r['success_rate'],
            'mean_length': r['mean_length'],
            'interaction_frequency': r.get('interaction_frequency', 0.0),
            'source': r.get('source', 'models'),
            'action_entropy': r.get('action_entropy', 0.0),
            'action_dist_up': r.get('action_distribution', {}).get('2', 0.0),
            'action_dist_down': r.get('action_distribution', {}).get('3', 0.0),
            'action_dist_left': r.get('action_distribution', {}).get('0', 0.0),
            'action_dist_right': r.get('action_distribution', {}).get('1', 0.0),
            'action_dist_interact': r.get('action_distribution', {}).get('4', 0.0),
            'learning_rate': r.get('training_config', {}).get('learning_rate', 0.0),
            'gamma': r.get('training_config', {}).get('gamma', 0.0),
            'entropy_coeff': r.get('training_config', {}).get('entropy_coeff', 0.0),
            'train_batch_size': r.get('training_config', {}).get('train_batch_size', 0),
            'num_epochs': r.get('training_config', {}).get('num_epochs', 0),
            'max_eval_reward': max(r.get('eval_rewards', [0.0])),
            'min_eval_reward': min(r.get('eval_rewards', [0.0])),
            'max_deliveries': max(r.get('deliveries_per_episode', [0])),
            'std_deliveries': float(np.std(r.get('deliveries_per_episode', [0]))),
            'std_length': float(np.std(r.get('eval_lengths', [0]))),
            'model_name': r.get('model_name', ''),
        })
    
    df = pd.DataFrame(df_data)
    csv_path = output_dir / "evaluation_results.csv"
    df.to_csv(csv_path, index=False)
    
    print(f"CSV saved in: {csv_path}")
    print(f"\nResults summary:")
    print(df.to_string(index=False))


In [ ]:
def visualize_model_behavior(checkpoint_path: str, num_episodes: int = 1, max_steps: int = 800, 
                             save_frames: bool = False, use_pygame: bool = True):
    """
    Visualizes model behavior using interactive pygame.
    
    Args:
        checkpoint_path: Path to the model checkpoint (string)
        num_episodes: Number of episodes to visualize (default: 1)
        max_steps: Maximum number of steps per episode
        save_frames: If True, saves frames to PNG files (default: False - visualization only)
        use_pygame: If True, shows interactive pygame visualization during execution
    
    Returns:
        Dictionary with collected metrics
    """
    from pathlib import Path
    
    # Load model
    algorithm = load_model_from_checkpoint(checkpoint_path)
    
    # Get policy
    shared_policy = None
    try:
        shared_policy = algorithm.get_policy("shared_policy")
    except Exception:
        try:
            if hasattr(algorithm, 'workers') and algorithm.workers:
                worker = algorithm.workers.local_worker()
                if hasattr(worker, 'get_policy'):
                    shared_policy = worker.get_policy("shared_policy")
                    if shared_policy is None:
                        policies = worker.get_policies()
                        if policies:
                            shared_policy = policies[list(policies.keys())[0]]
        except Exception:
            pass
    
    if shared_policy is None:
        raise RuntimeError("Could not get policy from algorithm.")
    
    # Create environment with rendering
    def env_creator_visualization():
        return OvercookedMARL(
            render_mode="human",
            map_path=ENVIRONMENT_CONFIG['map_path'],
            num_fixed_agents=ENVIRONMENT_CONFIG['num_fixed_agents'],
            verbose=False
        )
    
    env = ParallelEnvToMultiAgent(env_creator_visualization())
    
    all_frames = [] if save_frames else None
    all_metrics = []
    
    # Extract model name from checkpoint_path
    checkpoint_name = Path(checkpoint_path).name
    model_name = checkpoint_name.replace('checkpoint_', '').replace('/', '_')
    
    try:
        for episode in range(num_episodes):
            obs_dict, infos = env.reset()
            
            episode_frames = [] if save_frames else None
            episode_reward = 0.0
            episode_steps = 0
            
            # Count initial deliveries
            initial_delivered = 0
            if "agent_0" in infos and isinstance(infos["agent_0"], dict):
                initial_delivered = len(infos["agent_0"].get('delivered_recipes', []))
            
            # Render initial frame only if saving frames
            if save_frames:
                try:
                    frame = env.render()
                    if frame is not None and isinstance(frame, np.ndarray):
                        if len(frame.shape) == 3 and frame.shape[2] == 3:
                            episode_frames.append({
                                'frame': frame,
                                'step': 0,
                                'reward': 0.0,
                                'action': None,
                                'deliveries': 0
                            })
                except Exception:
                    pass
            else:
                # Just render to show pygame (without collecting)
                env.render()
            
            for step in range(max_steps):
                actions = {}
                
                for agent_id in env.agents:
                    try:
                        obs = obs_dict[agent_id]
                        
                        if not isinstance(obs, np.ndarray):
                            obs = np.array(obs)
                        
                        if obs.dtype == np.uint8:
                            obs_for_model = obs
                        elif obs.dtype == np.float32 or obs.dtype == np.float64:
                            if obs.max() <= 1.0:
                                obs_for_model = (obs * 255.0).astype(np.uint8)
                            else:
                                obs_for_model = np.clip(obs, 0, 255).astype(np.uint8)
                        else:
                            obs_for_model = obs.astype(np.uint8)
                        
                        action, state_out, info = shared_policy.compute_single_action(
                            obs=obs_for_model,
                            explore=True,
                        )
                        
                        if isinstance(action, (np.ndarray, list)):
                            action = int(action[0] if len(action) > 0 else action)
                        elif hasattr(action, 'item'):
                            action = int(action.item())
                        else:
                            action = int(action)
                        
                        actions[agent_id] = action
                        
                    except Exception as e:
                        actions[agent_id] = env.action_space[agent_id].sample()
                
                # Step no ambiente
                obs_dict, rewards, terminations, truncations, infos = env.step(actions)
                
                # Accumulate reward
                total_step_reward = sum(rewards.values())
                episode_reward += total_step_reward
                episode_steps += 1
                
                # Render frame (pygame is already active with render_mode="human")
                if save_frames:
                    # Collect frame only if saving
                    try:
                        frame = env.render()
                        if frame is not None and isinstance(frame, np.ndarray):
                            if len(frame.shape) == 3 and frame.shape[2] == 3:
                                current_deliveries = 0
                                if "agent_0" in infos and isinstance(infos["agent_0"], dict):
                                    current_deliveries = len(infos["agent_0"].get('delivered_recipes', [])) - initial_delivered
                                
                                episode_frames.append({
                                    'frame': frame,
                                    'step': step + 1,
                                    'reward': episode_reward,
                                    'action': actions.get('agent_0', None),
                                    'deliveries': current_deliveries
                                })
                    except Exception:
                        pass
                else:
                    # Just render to show pygame (without collecting)
                    env.render()
                
                # Check if episode ended
                if terminations.get('__all__', False) or truncations.get('__all__', False):
                    break
            
            # Count final deliveries
            final_delivered = 0
            if "agent_0" in infos and isinstance(infos["agent_0"], dict):
                final_delivered = len(infos["agent_0"].get('delivered_recipes', []))
            
            episode_deliveries = final_delivered - initial_delivered
            
            if save_frames:
                all_frames.append(episode_frames)
            
            all_metrics.append({
                'episode': episode,
                'reward': episode_reward,
                'steps': episode_steps,
                'deliveries': episode_deliveries
            })
            
    except KeyboardInterrupt:
        pass
    finally:
        env.close()
    
    if save_frames and all_frames:
        output_dir = Path("analysis_output")
        output_dir.mkdir(exist_ok=True)
        
        for ep_idx, (ep_frames, ep_metrics) in enumerate(zip(all_frames, all_metrics)):
            if not ep_frames:
                continue
            
            num_frames = len(ep_frames)
            fig, axes = plt.subplots(1, num_frames, figsize=(2 * num_frames, 4))
            
            if num_frames == 1:
                axes = [axes]
            
            for frame_idx, frame_data in enumerate(ep_frames):
                ax = axes[frame_idx]
                frame = frame_data['frame'] if isinstance(frame_data, dict) else frame_data
                ax.imshow(frame)
                ax.axis('off')
                
                title_parts = [f"Step {frame_data.get('step', 0)}"]
                if frame_data.get('reward') is not None:
                    title_parts.append(f"R: {frame_data['reward']:.1f}")
                if frame_data.get('deliveries') is not None:
                    title_parts.append(f"D: {frame_data['deliveries']}")
                ax.set_title(' | '.join(title_parts), fontsize=8)
            
            plt.suptitle(f"Episode {ep_metrics['episode']+1} - {num_frames} steps | Reward: {ep_metrics['reward']:.1f} | Deliveries: {ep_metrics['deliveries']}", 
                        fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.savefig(output_dir / f"visualization_{model_name}_ep{ep_metrics['episode']+1}.png", dpi=150, bbox_inches='tight')
            plt.show()
    
    return all_metrics


# Visualization

In [ ]:
# Define output directory
OUTPUT_DIR = Path("analysis_output")
OUTPUT_DIR.mkdir(exist_ok=True)

results_file = OUTPUT_DIR / "evaluation_results.json"

if results_file.exists():
    with open(results_file, 'r') as f:
        json_data = json.load(f)
    # JSON is saved with key 'results', not 'evaluation_results'
    evaluation_results = json_data.get('results', json_data.get('evaluation_results', []))
    
    if evaluation_results:
        df = pd.DataFrame(evaluation_results)
        df = df.sort_values(['mean_deliveries', 'mean_reward'], ascending=[False, False])
        
        best_model = df.iloc[0]
        checkpoint_path = best_model.get('checkpoint_path') or best_model.get('model_path')
        
        if checkpoint_path:
            n_episodes = 1
            
            visualize_model_behavior(
                checkpoint_path=str(checkpoint_path),
                num_episodes=n_episodes,
                max_steps=800,
                save_frames=False,  # Don't save frames - quick visualization only
                use_pygame=True  # Show pygame during execution
            )
        else:
            print(f"WARNING: checkpoint_path not found for model {best_model.get('config_name', 'unknown')}")
    else:
        print("No evaluation results found in JSON file")
else:
    # If no results file, discover models and evaluate
    models = discover_ray_checkpoints()
    models_from_models_dir = discover_models()
    models.extend(models_from_models_dir)
    
    if models:
        print("Evaluation results file not found. Evaluating all models first...")
        evaluation_results = []
        
        for model_info in tqdm(models, desc="Evaluating"):
            checkpoint_path = model_info.get('checkpoint_path')
            if checkpoint_path:
                result = evaluate_model(str(checkpoint_path), num_episodes=15)
                if result:
                    evaluation_results.append(result)
        
        if evaluation_results:
            df = pd.DataFrame(evaluation_results)
            df = df.sort_values(['mean_deliveries', 'mean_reward'], ascending=[False, False])
            
            best_model = df.iloc[0]
            checkpoint_path = best_model.get('checkpoint_path') or best_model.get('model_path')
            
            if checkpoint_path:
                n_episodes = 1
                
                visualize_model_behavior(
                    checkpoint_path=str(checkpoint_path),
                    num_episodes=n_episodes,
                    max_steps=800,
                    save_frames=False,  # Don't save frames - quick visualization only
                    use_pygame=True  # Show pygame during execution
                )
            else:
                print(f"WARNING: checkpoint_path not found for model {best_model.get('config_name', 'unknown')}")
        else:
            print("No models could be evaluated")
    else:
        print("No models found in ray_results/ or models/ directory")
